# PCB Anomaly Detection with PatchCore

This notebook provides a complete image-based PCB anomaly detection pipeline:

**Input → PCB Alignment → Image-level Split → Tiling → PatchCore Training → Threshold Calibration → Holdout Evaluation → Heatmap → Inference**

The pipeline is designed to work with a local project structure and does not depend on a specific cloud platform.

## 1. Dependencies

In [1]:
# Install the required Python package.
!pip install -q "anomalib==2.6.0" opencv-python pandas matplotlib

## 2. Configuration

In [2]:
# =========================================================
# CONFIGURATION
# =========================================================

from pathlib import Path
import json
import random
import shutil

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from anomalib.models import Patchcore
from anomalib.engine import Engine
from anomalib.data import Folder, PredictDataset

# Input and output directories.
RAW_DATA_DIR = Path("raw_data")
REFERENCE_NAME = "Healthy.png"
OUTPUT_ROOT = Path("outputs/patchcore_run")

# Alignment parameters.
ORB_FEATURES = 8000
ORB_RATIO_TEST = 0.75
MIN_GOOD_MATCHES = 20
MIN_INLIERS = 12
MIN_INLIER_RATIO = 0.25
MAX_REPROJ_ERROR = 6.0
RANSAC_REPROJ_THRESHOLD = 4.0

# Tiling parameters.
TILE_SIZE = 512
TILE_OVERLAP = 128
TILE_STEP = TILE_SIZE - TILE_OVERLAP

# Image-level dataset split.
RANDOM_SEED = 42
TRAIN_RATIO = 0.70
CALIBRATION_RATIO = 0.15
HOLDOUT_RATIO = 0.15

# PatchCore parameters.
BACKBONE = "wide_resnet50_2"
LAYERS = ["layer2", "layer3"]
CORESET_RATIO = 0.01
NUM_NEIGHBORS = 9
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
NUM_WORKERS = 2

# Image-level score and threshold settings.
TOP_K_TILES = 2
CALIBRATION_QUANTILE = 0.99

# Optional stage controls.
RUN_ALIGNMENT = True
RUN_TILING = True
RUN_TRAINING = True
RUN_CALIBRATION = True
RUN_HOLDOUT_CHECK = True

print("Raw data:", RAW_DATA_DIR)
print("Reference:", RAW_DATA_DIR / REFERENCE_NAME)
print("Output:", OUTPUT_ROOT)

d:\cods\DataSince\VENVS\Dl_venv\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
d:\cods\DataSince\VENVS\Dl_venv\lib\site-packages\anomalib\models\image\dinomaly\components\layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention


Raw data: raw_data
Reference: raw_data\Healthy.png
Output: outputs\patchcore_run


## 3. Input Validation and Output Structure

In [3]:
# Validate the input directory and reference image.
reference_path = RAW_DATA_DIR / REFERENCE_NAME

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(f"Raw data directory not found: {RAW_DATA_DIR}")
if not reference_path.exists():
    raise FileNotFoundError(f"Reference image not found: {reference_path}")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
raw_images = sorted(
    p for p in RAW_DATA_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS and p.name != REFERENCE_NAME
)

# Create the output structure used by later stages.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ALIGN_DIR = OUTPUT_ROOT / "aligned"
ALIGN_FAIL_DIR = OUTPUT_ROOT / "alignment_failed"
QC_DIR = OUTPUT_ROOT / "qc_alignment"
TILE_ROOT = OUTPUT_ROOT / "tiles"
DATASET_ROOT = OUTPUT_ROOT / "anomalib_dataset"
TRAIN_DIR = DATASET_ROOT / "train" / "good"
CALIB_DIR = DATASET_ROOT / "calibration" / "good"
HOLDOUT_DIR = OUTPUT_ROOT / "holdout_tiles"
RESULTS_DIR = OUTPUT_ROOT / "results"

for directory in [
    ALIGN_DIR, ALIGN_FAIL_DIR, QC_DIR,
    TRAIN_DIR, CALIB_DIR, HOLDOUT_DIR, RESULTS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Raw images: {len(raw_images)}")

Raw images: 88


## 4. PCB Alignment

In [4]:
# =========================================================
# IMAGE ALIGNMENT
# Detect, crop, rotate, and align each PCB to the reference.
# =========================================================

def load_bgr(path: Path):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Failed to read image: {path}")
    return image


def order_points(pts):
    pts = np.asarray(pts, dtype=np.float32)
    s = pts.sum(axis=1)
    d = np.diff(pts, axis=1).ravel()
    return np.array([
        pts[np.argmin(s)], pts[np.argmin(d)],
        pts[np.argmax(s)], pts[np.argmax(d)]
    ], dtype=np.float32)


def detect_green_pcb_mask(image):
    # Detect green PCB regions under different lighting conditions.
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    masks = [
        cv2.inRange(hsv, np.array([30, 35, 20]), np.array([100, 255, 255])),
        cv2.inRange(hsv, np.array([35, 60, 30]), np.array([90, 255, 255]))
    ]

    best = None
    close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (21, 21))
    open_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

    for mask in masks:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, close_kernel, iterations=2)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, open_kernel)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)
        area = float(cv2.contourArea(contour))
        if best is None or area > best[0]:
            best = (area, contour, mask)

    if best is None:
        return None, None

    area, contour, mask = best
    if area < image.shape[0] * image.shape[1] * 0.03:
        return None, None
    return mask, contour


def crop_and_rotate_board(image):
    # Extract the PCB and normalize its orientation.
    mask, contour = detect_green_pcb_mask(image)
    if contour is None:
        return None, None

    box = cv2.boxPoints(cv2.minAreaRect(contour)).astype(np.float32)
    box = order_points(box)
    width_a = np.linalg.norm(box[2] - box[3])
    width_b = np.linalg.norm(box[1] - box[0])
    height_a = np.linalg.norm(box[1] - box[2])
    height_b = np.linalg.norm(box[0] - box[3])
    crop_w = max(10, int(round(max(width_a, width_b))))
    crop_h = max(10, int(round(max(height_a, height_b))))

    dst = np.array([
        [0, 0], [crop_w - 1, 0],
        [crop_w - 1, crop_h - 1], [0, crop_h - 1]
    ], dtype=np.float32)
    M = cv2.getPerspectiveTransform(box, dst)
    crop = cv2.warpPerspective(
        image, M, (crop_w, crop_h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE
    )
    if crop.shape[0] > crop.shape[1]:
        crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)
    return crop, mask


def orb_homography_align(reference, board_crop):
    # Align the cropped PCB to the reference using ORB + RANSAC Homography.
    ref_gray = cv2.cvtColor(reference, cv2.COLOR_BGR2GRAY)
    img_gray = cv2.cvtColor(board_crop, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(nfeatures=ORB_FEATURES, fastThreshold=10, edgeThreshold=15)
    kp_ref, des_ref = orb.detectAndCompute(ref_gray, None)
    kp_img, des_img = orb.detectAndCompute(img_gray, None)

    if des_ref is None or des_img is None:
        return None, {"status": "no_descriptors", "good_matches": 0, "inliers": 0, "inlier_ratio": 0.0, "reproj_error": np.nan}

    matches = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(des_img, des_ref, k=2)
    good = [m for pair in matches if len(pair) == 2 for m, n in [pair] if m.distance < ORB_RATIO_TEST * n.distance]
    good.sort(key=lambda m: m.distance)

    if len(good) < MIN_GOOD_MATCHES:
        return None, {"status": "too_few_good_matches", "good_matches": len(good), "inliers": 0, "inlier_ratio": 0.0, "reproj_error": np.nan}

    src = np.float32([kp_img[m.queryIdx].pt for m in good])
    dst = np.float32([kp_ref[m.trainIdx].pt for m in good])
    H, inlier_mask = cv2.findHomography(
        src, dst, cv2.RANSAC, RANSAC_REPROJ_THRESHOLD,
        maxIters=5000, confidence=0.995
    )
    if H is None or inlier_mask is None:
        return None, {"status": "homography_failed", "good_matches": len(good), "inliers": 0, "inlier_ratio": 0.0, "reproj_error": np.nan}

    inliers = inlier_mask.ravel().astype(bool)
    inlier_count = int(inliers.sum())
    inlier_ratio = inlier_count / max(len(good), 1)
    projected = cv2.perspectiveTransform(src[inliers].reshape(-1, 1, 2), H).reshape(-1, 2)
    reproj_error = float(np.mean(np.linalg.norm(projected - dst[inliers], axis=1))) if len(projected) else np.inf

    report = {"status": "ok", "good_matches": len(good), "inliers": inlier_count, "inlier_ratio": inlier_ratio, "reproj_error": reproj_error}
    if inlier_count < MIN_INLIERS or inlier_ratio < MIN_INLIER_RATIO or reproj_error > MAX_REPROJ_ERROR:
        report["status"] = "weak_alignment"
        return None, report

    aligned = cv2.warpPerspective(
        board_crop, H, (reference.shape[1], reference.shape[0]),
        flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE
    )
    return aligned, report


def align_one_image(reference, image_path):
    # Run the complete alignment pipeline for one image.
    image = load_bgr(image_path)
    board_crop, _ = crop_and_rotate_board(image)
    if board_crop is None:
        return None, {"status": "pcb_crop_failed", "good_matches": 0, "inliers": 0, "inlier_ratio": 0.0, "reproj_error": np.nan}, None
    aligned, report = orb_homography_align(reference, board_crop)
    return aligned, report, board_crop

In [ ]:
# Apply alignment to the full dataset.
if RUN_ALIGNMENT:
    reference = load_bgr(reference_path)
    alignment_rows, qc_items = [], []

    for idx, image_path in enumerate(raw_images, 1):
        try:
            aligned, report, _ = align_one_image(reference, image_path)
            alignment_rows.append({
                "filename": str(image_path.relative_to(RAW_DATA_DIR)),
                **report
            })

            if aligned is not None:
                cv2.imwrite(str(ALIGN_DIR / image_path.name), aligned, [cv2.IMWRITE_JPEG_QUALITY, 95])
                if len(qc_items) < 12:
                    overlay = cv2.addWeighted(reference, 0.5, aligned, 0.5, 0)
                    qc_items.append((image_path.name, aligned, overlay, report))
            else:
                shutil.copy2(image_path, ALIGN_FAIL_DIR / image_path.name)

            print(f"[{idx}/{len(raw_images)}] {image_path.name} -> {report['status']}")
        except Exception as exc:
            alignment_rows.append({
                "filename": str(image_path.relative_to(RAW_DATA_DIR)),
                "status": f"exception:{type(exc).__name__}",
                "good_matches": 0, "inliers": 0, "inlier_ratio": 0.0, "reproj_error": np.nan
            })
            shutil.copy2(image_path, ALIGN_FAIL_DIR / image_path.name)
            print(f"ERROR: {image_path.name}: {exc}")

    alignment_df = pd.DataFrame(alignment_rows)
    alignment_df.to_csv(OUTPUT_ROOT / "alignment_report.csv", index=False)
    print("\nAlignment summary:")
    print(alignment_df["status"].value_counts(dropna=False))

## 5. Image-level Dataset Split

In [ ]:
# Keep only successfully aligned images and split them before tiling.
ok_df = alignment_df[alignment_df["status"] == "ok"].copy()
if len(ok_df) < 12:
    raise RuntimeError(f"Only {len(ok_df)} aligned images are available; at least 12 are required.")

ok_paths = [ALIGN_DIR / Path(name).name for name in ok_df["filename"]]
random.Random(RANDOM_SEED).shuffle(ok_paths)
n = len(ok_paths)
n_train = max(1, int(round(n * TRAIN_RATIO)))
n_calib = max(1, int(round(n * CALIBRATION_RATIO)))
while n_train + n_calib >= n:
    if n_train > n_calib: n_train -= 1
    else: n_calib -= 1

train_paths = ok_paths[:n_train]
calib_paths = ok_paths[n_train:n_train + n_calib]
holdout_paths = ok_paths[n_train + n_calib:]

split_df = pd.DataFrame(
    [{"filename": p.name, "split": split}
     for split, paths in [("train", train_paths), ("calibration", calib_paths), ("holdout", holdout_paths)]
     for p in paths]
)
split_df.to_csv(OUTPUT_ROOT / "image_split.csv", index=False)

print(f"Train: {len(train_paths)}")
print(f"Calibration: {len(calib_paths)}")
print(f"Holdout: {len(holdout_paths)}")

## 6. Tiling

In [7]:
# =========================================================
# IMAGE TILING
# =========================================================

def pad_to_tile_size(image, tile_size=TILE_SIZE):
    # Add reflected padding when image height is smaller than the tile size.
    h, _ = image.shape[:2]
    pad_h = max(0, tile_size - h)
    top = pad_h // 2
    bottom = pad_h - top
    if pad_h == 0:
        return image, 0, 0
    padded = cv2.copyMakeBorder(image, top, bottom, 0, 0, cv2.BORDER_REFLECT_101)
    return padded, top, bottom


def tile_positions(width, height, tile_size=TILE_SIZE, overlap=TILE_OVERLAP):
    # Generate sliding-window start positions and ensure edge coverage.
    step = tile_size - overlap
    xs = list(range(0, max(1, width - tile_size + 1), step))
    ys = list(range(0, max(1, height - tile_size + 1), step))
    if width >= tile_size and (not xs or xs[-1] != width - tile_size): xs.append(width - tile_size)
    if height >= tile_size and (not ys or ys[-1] != height - tile_size): ys.append(height - tile_size)
    return xs, ys


def write_tiles_for_image(image_path, output_dir, split_name, image_id):
    # Save overlapping tiles and return their metadata.
    image = load_bgr(image_path)
    padded, pad_top, pad_bottom = pad_to_tile_size(image)
    h, w = padded.shape[:2]
    xs, ys = tile_positions(w, h)
    output_dir.mkdir(parents=True, exist_ok=True)
    rows, tile_index = [], 0

    for y in ys:
        for x in xs:
            tile = padded[y:y + TILE_SIZE, x:x + TILE_SIZE]
            path = output_dir / f"{image_id}_tile_{tile_index:03d}_x{x}_y{y}.jpg"
            cv2.imwrite(str(path), tile, [cv2.IMWRITE_JPEG_QUALITY, 95])
            rows.append({"image_id": image_id, "source_image": image_path.name, "split": split_name, "tile_index": tile_index, "x": x, "y": y, "tile_size": TILE_SIZE, "padding_top": pad_top, "padding_bottom": pad_bottom, "tile_path": str(path)})
            tile_index += 1
    return rows

In [ ]:
# Generate tiles for train, calibration, and holdout sets.
if RUN_TILING:
    for directory in [TRAIN_DIR, CALIB_DIR, HOLDOUT_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
        for file in directory.glob("*.jpg"):
            file.unlink()

    all_tile_rows = []
    for split_name, paths, output_dir in [
        ("train", train_paths, TRAIN_DIR),
        ("calibration", calib_paths, CALIB_DIR),
        ("holdout", holdout_paths, HOLDOUT_DIR),
    ]:
        for image_path in paths:
            all_tile_rows.extend(write_tiles_for_image(image_path, output_dir, split_name, image_path.stem))

    manifest_df = pd.DataFrame(all_tile_rows)
    manifest_df.to_csv(OUTPUT_ROOT / "tile_manifest.csv", index=False)
    print("Total tiles:", len(manifest_df))
    print(manifest_df.groupby(["split", "image_id"]).size().head(20))

In [ ]:
# Quick visual check of one generated tile.
sample_path = Path(manifest_df.iloc[0]["tile_path"])
sample = load_bgr(sample_path)
plt.figure(figsize=(5, 5))
plt.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
plt.title(f"Sample tile: {sample_path.name}")
plt.axis("off")
plt.show()

## 7. PatchCore Training

In [ ]:
# Train PatchCore using healthy training tiles.
if not TRAIN_DIR.exists() or not any(TRAIN_DIR.glob("*.jpg")):
    raise RuntimeError("Training tiles were not found.")

model = Patchcore(
    backbone=BACKBONE,
    layers=LAYERS,
    pre_trained=True,
    coreset_sampling_ratio=CORESET_RATIO,
    num_neighbors=NUM_NEIGHBORS,
)

datamodule = Folder(
    name="PCB_Anomaly_Detection",
    root=str(DATASET_ROOT),
    normal_dir="train/good",
    normal_test_dir="calibration/good",
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
engine = Engine(
    accelerator=accelerator,
    devices=1,
    max_epochs=1,
    default_root_dir=str(RESULTS_DIR),
    enable_progress_bar=False,
)

print("Device:", accelerator)
print("Training tiles:", len(list(TRAIN_DIR.glob("*.jpg"))))
print("Calibration tiles:", len(list(CALIB_DIR.glob("*.jpg"))))

if RUN_TRAINING:
    engine.fit(model=model, datamodule=datamodule)

    best_path = ""
    try:
        best_path = engine.trainer.checkpoint_callback.best_model_path
    except Exception:
        pass

    if not best_path:
        fallback = RESULTS_DIR / "patchcore_checkpoint.ckpt"
        engine.trainer.save_checkpoint(str(fallback))
        best_path = str(fallback)

    FINAL_CKPT = OUTPUT_ROOT / "patchcore_model.ckpt"
    if Path(best_path).resolve() != FINAL_CKPT.resolve():
        shutil.copy2(best_path, FINAL_CKPT)
    print("Final checkpoint:", FINAL_CKPT)

## 8. Tile-level Prediction and Image-level Scoring

In [ ]:
# Predict tile scores and anomaly maps with the trained checkpoint.

def prediction_scalar(x):
    try:
        return float(x.detach().cpu().reshape(-1)[0])
    except Exception:
        return float(x)


def predict_tile_folder(tile_dir: Path, checkpoint_path: Path):
    model_for_pred = Patchcore(
        backbone=BACKBONE,
        layers=LAYERS,
        pre_trained=True,
        coreset_sampling_ratio=CORESET_RATIO,
        num_neighbors=NUM_NEIGHBORS,
    )
    predict_engine = Engine(
        accelerator=accelerator,
        devices=1,
        enable_progress_bar=False,
    )
    dataset = PredictDataset(
        path=str(tile_dir),
        image_size=(256, 256),
    )
    predictions = predict_engine.predict(
        model=model_for_pred,
        dataset=dataset,
        ckpt_path=str(checkpoint_path),
    )
    if predictions is None:
        return pd.DataFrame(columns=["image_id", "tile_name", "x", "y", "score", "anomaly_map"])

    rows = []
    for pred in predictions:
        image_path = Path(str(pred.image_path))
        stem = image_path.stem
        try:
            x = int(stem.split("_x")[-1].split("_y")[0])
            y = int(stem.split("_y")[-1])
        except Exception:
            x = y = 0
        rows.append({
            "image_id": stem.rsplit("_tile_", 1)[0],
            "tile_name": image_path.name,
            "x": x,
            "y": y,
            "score": prediction_scalar(pred.pred_score),
            "anomaly_map": pred.anomaly_map.detach().cpu().numpy().squeeze(),
        })
    return pd.DataFrame(rows)


def image_level_scores(tile_df, top_k=TOP_K_TILES):
    # Aggregate tile scores into one score per source image.
    rows = []
    for image_id, group in tile_df.groupby("image_id"):
        scores = np.sort(group["score"].to_numpy())
        k = min(top_k, len(scores))
        rows.append({
            "image_id": image_id,
            "image_score": float(scores[-k:].mean()),
            "max_tile_score": float(scores.max()),
            "mean_tile_score": float(scores.mean()),
            "num_tiles": len(scores),
        })
    return pd.DataFrame(rows)

## 9. Threshold Calibration

In [ ]:
# Calculate a threshold from healthy calibration images.
if RUN_CALIBRATION:
    calib_pred_df = predict_tile_folder(CALIB_DIR, FINAL_CKPT)
    if calib_pred_df.empty:
        raise RuntimeError("No calibration predictions were returned.")

    calib_image_scores = image_level_scores(calib_pred_df)
    threshold = float(calib_image_scores["image_score"].quantile(CALIBRATION_QUANTILE))
    calib_image_scores["predicted_anomaly"] = calib_image_scores["image_score"] > threshold
    calib_image_scores.to_csv(OUTPUT_ROOT / "calibration_scores.csv", index=False)

    with open(OUTPUT_ROOT / "threshold.json", "w", encoding="utf-8") as file:
        json.dump({
            "method": "top_k_tile_mean",
            "top_k": TOP_K_TILES,
            "quantile": CALIBRATION_QUANTILE,
            "threshold": threshold,
        }, file, indent=2)

    print(f"Threshold (quantile={CALIBRATION_QUANTILE}): {threshold:.6f}")
    display(calib_image_scores.sort_values("image_score", ascending=False))

## 10. Holdout Evaluation

In [ ]:
# Evaluate false positives on unseen healthy holdout images.
if RUN_HOLDOUT_CHECK:
    holdout_pred_df = predict_tile_folder(HOLDOUT_DIR, FINAL_CKPT)
    holdout_image_scores = image_level_scores(holdout_pred_df)
    holdout_image_scores["predicted_anomaly"] = holdout_image_scores["image_score"] > threshold
    holdout_image_scores.to_csv(OUTPUT_ROOT / "holdout_scores.csv", index=False)

    false_positives = int(holdout_image_scores["predicted_anomaly"].sum())
    total = len(holdout_image_scores)
    fpr = false_positives / total if total else np.nan

    print("Holdout healthy images:", total)
    print("False positives:", false_positives)
    print(f"False positive rate: {fpr:.3f}")
    display(holdout_image_scores.sort_values("image_score", ascending=False))

## 11. Heatmap Generation

In [ ]:
# Reconstruct a full-image anomaly heatmap from tile-level maps.

def build_full_heatmap_from_predictions(aligned_image, tile_pred_df):
    padded, pad_top, pad_bottom = pad_to_tile_size(aligned_image)
    ph, pw = padded.shape[:2]
    heat_sum = np.zeros((ph, pw), dtype=np.float32)
    heat_count = np.zeros((ph, pw), dtype=np.float32)

    for _, row in tile_pred_df.iterrows():
        x, y = int(row["x"]), int(row["y"])
        resized = cv2.resize(row["anomaly_map"], (TILE_SIZE, TILE_SIZE), interpolation=cv2.INTER_LINEAR)
        end_y, end_x = min(y + TILE_SIZE, ph), min(x + TILE_SIZE, pw)
        rh, rw = end_y - y, end_x - x
        heat_sum[y:end_y, x:end_x] += resized[:rh, :rw]
        heat_count[y:end_y, x:end_x] += 1.0

    padded_heat = heat_sum / np.maximum(heat_count, 1.0)
    if pad_top + pad_bottom > 0:
        return padded_heat[pad_top:pad_top + aligned_image.shape[0], :aligned_image.shape[1]]
    return padded_heat[:aligned_image.shape[0], :aligned_image.shape[1]]


def show_heatmap_overlay(image, heatmap, title="PatchCore Heatmap"):
    # Convert anomaly values to a color map and overlay it on the PCB.
    normalized = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    color_map = cv2.applyColorMap(normalized, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(image, 0.55, color_map, 0.45, 0)

    plt.figure(figsize=(18, 6))
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()
    return overlay

## 12. Calibration Heatmap QC

In [ ]:
# Visualize the highest-scoring healthy calibration image.
example_image_id = calib_image_scores.sort_values("image_score", ascending=False).iloc[0]["image_id"]
example_tiles = calib_pred_df[calib_pred_df["image_id"] == example_image_id].copy()
example_aligned = load_bgr(ALIGN_DIR / f"{example_image_id}.jpg")
example_heat = build_full_heatmap_from_predictions(example_aligned, example_tiles)
example_overlay = show_heatmap_overlay(example_aligned, example_heat, f"Calibration example — {example_image_id}")

example_output = QC_DIR / f"heatmap_{example_image_id}.jpg"
cv2.imwrite(str(example_output), example_overlay)
print("Example heatmap:", example_output)

## 13. Final Inference

In [ ]:
# Run the complete pipeline on one new raw PCB image.
def infer_raw_board(raw_image_path: str | Path):
    raw_image_path = Path(raw_image_path)
    if not raw_image_path.exists():
        raise FileNotFoundError(raw_image_path)

    aligned, report, _ = align_one_image(reference, raw_image_path)
    if aligned is None:
        raise RuntimeError(f"Alignment failed: {report}")

    temp_dir = OUTPUT_ROOT / "inference_temp"
    temp_dir.mkdir(parents=True, exist_ok=True)
    for old in temp_dir.glob("*.jpg"):
        old.unlink()

    aligned_temp = temp_dir / f"{raw_image_path.stem}_aligned.jpg"
    cv2.imwrite(str(aligned_temp), aligned, [cv2.IMWRITE_JPEG_QUALITY, 95])
    write_tiles_for_image(aligned_temp, temp_dir, "inference", raw_image_path.stem)

    pred_df = predict_tile_folder(temp_dir, FINAL_CKPT)
    if pred_df.empty:
        raise RuntimeError("No prediction was returned.")

    image_score = float(image_level_scores(pred_df).iloc[0]["image_score"])
    decision = image_score > threshold
    heat = build_full_heatmap_from_predictions(aligned, pred_df)
    overlay = show_heatmap_overlay(
        aligned, heat,
        title=f"{raw_image_path.name} | score={image_score:.5f} | anomaly={decision}"
    )

    output_path = RESULTS_DIR / f"{raw_image_path.stem}_result.jpg"
    cv2.imwrite(str(output_path), overlay)

    result = {
        "image": str(raw_image_path),
        "alignment_status": report["status"],
        "good_matches": report["good_matches"],
        "inliers": report["inliers"],
        "inlier_ratio": report["inlier_ratio"],
        "reproj_error": report["reproj_error"],
        "image_score": image_score,
        "threshold": threshold,
        "predicted_anomaly": bool(decision),
        "output": str(output_path),
    }
    print(json.dumps(result, indent=2, default=str))
    return result


# Batch inference on all images in a folder.
TEST_FOLDER = Path("test")

image_files = sorted(
    p for p in TEST_FOLDER.iterdir()
    if p.suffix.lower() in IMAGE_EXTS
) if TEST_FOLDER.exists() else []

if image_files:
    results = []
    for image_path in image_files:
        try:
            result = infer_raw_board(image_path)
            results.append({
                "image": image_path.name,
                "score": result["image_score"],
                "anomaly": result["predicted_anomaly"],
                "output": result["output"],
            })
        except Exception as exc:
            print(f"ERROR: {image_path.name}: {exc}")

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_ROOT / "inference_results.csv", index=False)
    display(results_df)
else:
    print("Set TEST_FOLDER to a directory containing images, then run this cell again.")

## Output Files

Key outputs include:

- `aligned/` — successfully aligned PCB images
- `alignment_failed/` — images that failed alignment
- `alignment_report.csv` — alignment quality report
- `image_split.csv` — image-level train/calibration/holdout split
- `tiles/` and Anomalib dataset folders — generated tiles
- `tile_manifest.csv` — tile metadata
- `patchcore_model.ckpt` — trained PatchCore checkpoint
- `calibration_scores.csv` and `threshold.json` — calibrated threshold
- `holdout_scores.csv` — holdout evaluation
- `results/` — inference visualizations